In [1]:
print("python")

python


In [2]:
from keras.datasets import mnist
import numpy as np

In [3]:
import tensorflow as tf
import tensorflow_datasets as tfds

In [4]:
(X_train_raw,y_train_raw),(X_test_raw,y_test_raw)= mnist.load_data()

In [5]:
X_val_raw=X_test_raw[-10000:]
y_val_raw=y_test_raw[-10000:]

X_train_raw=X_train_raw[:-10000]
y_train_raw=y_train_raw[:-10000]

print(f"Train shape: {X_train_raw.shape}")
print(f"Val shape: {X_val_raw.shape}")  

Train shape: (50000, 28, 28)
Val shape: (10000, 28, 28)


In [6]:
# Images are 28x28 grids of pixels with values ranging from 0 to 255
# so I will flatten the 2D to 1D vectors (784 features)and scale the pixels to be between 0 and 1
# FLATTENING
X_train = X_train_raw.reshape(-1,784)/ 255.0
X_val=X_val_raw.reshape(-1,784)/255.0
X_test=X_test_raw.reshape(-1,784)/255.0

In [7]:
# One hot encode the labels 
def one_hot(y,num_classes=10):
    # np.eye creates an identity matrix, and indexing it with 'y'
    return np.eye(num_classes)[y]

y_train=one_hot(y_train_raw)
y_val=one_hot(y_val_raw)
y_test=one_hot(y_test_raw)


##### BUILDING A VANILLA MLP

In [8]:
#Parameter initialization 
def init_params():
    W1=np.random.randn(784,100) *0.1
    b1=np.zeros((1,100))

    W2=np.random.randn(100,100) * 0.1
    b2=np.zeros((1,100))
    
    W3= np.random.randn(100,10) *0.1
    b3 = np.zeros((1,10))

    return {'W1':W1, 'b1':b1,"W2":W2,"b2":b2,"W3":W3,"b3":b3}

In [9]:
def Tanh(Z):
    return np.tanh(Z)

def softmax(Z):
    exp_Z=np.exp(Z - np.max(Z, axis =1 , keepdims=True))
    return exp_Z/ np.sum(exp_Z, axis =1 , keepdims = True)

In [10]:
def foward_pass(X, params):
    # Layer 1 
    Z1=np.dot(X, params["W1"]) + params["b1"]
    A1= Tanh(Z1)

    # Layer 2
    Z2 =np.dot(A1,params["W2"]) + params["b2"]
    A2= Tanh(Z2)

    # Layer 3 
    Z3=np.dot(A2,params["W3"])+ params["b3"]
    A3 = softmax(Z3)

    cache={"Z1":Z1 , "A1":A1, "Z2":Z2, "A2":A2, "Z3": Z3 , "A3":A3}

    return A3, cache


In [11]:
def loss_function(Y_true,Y_pred):

    m= Y_true.shape[0] # number of examples in the batch 

    epsilon=1e-15
    y_pred_clipped= np.clip(Y_pred, epsilon, 1 - epsilon)
    loss = -np.sum(Y_true* np.log(y_pred_clipped))/m

    return loss


In [13]:
def backward_pass(X, Y, cache, params):
    m=X.shape[0] # Batch size

    # activation from the foward pass
    A1= cache["A1"]
    A2= cache["A2"]
    A3= cache["A3"]

    W2= params["W2"]
    W3 = params["W3"]

    # layer 3 the output
    dZ3= A3- Y
    dW3 = (1/m) * np.dot(A2.T, dZ3)
    db3=(1/m) * np.sum(dZ3, axis=0, keepdims=True)
    
    # the derivative of Tanh
    dA2= np.dot(dZ3 , W3.T)
    dZ2= dA2 * (1- np.power(A2, 2))
    dW2 =  (1/m) * np.dot(A1.T, dZ2)
    db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True)

    # Layer 1
    dA1= np.dot(dZ2, W2.T)
    dZ1= dA1 * (1-np.power(A1, 2))
    dW1= (1/m) * np.dot(X.T, dZ1)
    db1 = (1/m) * np.sum(dZ1, axis=0,keepdims=True)

    # Store the gradients in a dictionary 
    grads= {"dW1":dW1, "db1":db1, "dW2":dW2, "db2":db2, "dW3":dW3,"db3":db3} 

    return grads

    

In [29]:
def update_params(params, grads, learning_rate):
    '''this is the optimizer calculated as W=W-(Lr*dW) dW being the derivative of W, while 
        W is the parameter in question'''
    params["W1"] -= learning_rate * grads["dW1"]
    params["b1"] -= learning_rate * grads["db1"]
    params["W2"] -= learning_rate * grads["dW2"]
    params["b2"] -= learning_rate * grads["db2"]
    params["W3"] -= learning_rate * grads["dW3"]
    params["b3"] -= learning_rate * grads["db3"]

    return params 

In [33]:
# params = init_params()
# for i in range(20000):
#     A3,cache = foward_pass(X=X_train, params=params)
#     loss=loss_function(Y_true=y_train,Y_pred=A3)
#     grads= backward_pass(X=X_train,Y=y_train, cache=cache,params=params)
#     params=update_params(params=params, grads=grads, learning_rate=0.01)
    
steps=20000
batch_size=256
learning_rate=0.1

params= init_params()
m = X_train.shape[0]

for step in range(steps):
    # shuffling the data first 
    ix=np.random.randint(0,m, size=batch_size)
    X_batch=X_train[ix]
    y_batch=y_train[ix]

    learning_rate= 0.1 if step<10000 else 0.01
    # foward_pass
    A3,cache= foward_pass(X=X_batch,params= params)
    loss= loss_function(Y_true=y_batch, Y_pred=A3)
    grads=backward_pass(X=X_batch,Y= y_batch,cache=cache, params=params)
    params = update_params(params= params, grads=grads, learning_rate=learning_rate)
    if step % 1000 == 0:
        A3_val,_=foward_pass(X_val,params)
        val_loss= loss_function(y_val, A3_val)
        print(f"Step {step:5d} | Train Loss: {loss:.4f} | Val loss {val_loss:.4f}")

A3_test,_= foward_pass(X_test, params)
test_loss = loss_function(y_test,A3_test)

predictions= np.argmax(A3_test, axis=1)
true_labels=np.argmax(y_test, axis=1)

accuracy=np.mean(predictions== true_labels) * 100

print(f"--- VANILLA NETWORK FINAL RESULTS ---")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {accuracy:.2f}%")

Step     0 | Train Loss: 2.4890 | Val loss 2.3716
Step  1000 | Train Loss: 0.2079 | Val loss 0.2157
Step  2000 | Train Loss: 0.1701 | Val loss 0.1525
Step  3000 | Train Loss: 0.0899 | Val loss 0.1259
Step  4000 | Train Loss: 0.0421 | Val loss 0.1103
Step  5000 | Train Loss: 0.0424 | Val loss 0.0988
Step  6000 | Train Loss: 0.0530 | Val loss 0.0923
Step  7000 | Train Loss: 0.0399 | Val loss 0.0878
Step  8000 | Train Loss: 0.0390 | Val loss 0.0839
Step  9000 | Train Loss: 0.0299 | Val loss 0.0849
Step 10000 | Train Loss: 0.0264 | Val loss 0.0839
Step 11000 | Train Loss: 0.0213 | Val loss 0.0824
Step 12000 | Train Loss: 0.0238 | Val loss 0.0826
Step 13000 | Train Loss: 0.0123 | Val loss 0.0822
Step 14000 | Train Loss: 0.0202 | Val loss 0.0828
Step 15000 | Train Loss: 0.0218 | Val loss 0.0823
Step 16000 | Train Loss: 0.0127 | Val loss 0.0823
Step 17000 | Train Loss: 0.0173 | Val loss 0.0822
Step 18000 | Train Loss: 0.0164 | Val loss 0.0825
Step 19000 | Train Loss: 0.0096 | Val loss 0.0826


C:\Users\Admin\AppData\Local\Temp\ipykernel_17052\1152209723.py:1: RuntimeWarning: divide by zero encountered in log
  np.log(0).item()


-inf